# Regenerate the class-wise MLT table

Table 5 disagrees with Fig. 11(b) on every class by +1.6 to +15.7 mAP50 points, all in
one direction, and its implied overall mAP50 is 73.8 against 66.3 in the figure legend
and 66.1 in `results.csv`. Precision reconciles (80.3 vs 81.0) but recall and mAP do
not, so those columns did not come from the same evaluation as the rest of the paper.

This runs the evaluation again from `best.pt` and prints a replacement table.
Inference only, roughly two minutes.

Attach **`rishiksaisanthosh/dataset-test`**. It supplies both the validation split and `best.pt`, which
avoids the git-lfs pointer a plain clone would give you.


In [ ]:
!pip install -q "ultralytics==8.3.189"
!git clone -q --branch ablation-mscbam-probe https://github.com/SaiSanthosh1508/End-to-End-Text-Translation-Pipeline.git /kaggle/working/repo
!cd /kaggle/working/repo && python ablation/install_modules.py

In [ ]:
import pathlib

roots = sorted({
    hit.parent.parent
    for hit in pathlib.Path("/kaggle/input").glob("**/images/val")
    if (hit.parent.parent / "labels/val").is_dir()
})
if not roots:
    raise RuntimeError("attach the MLT dataset before running this")

root = max(roots, key=lambda r: len(list((r / "images/val").glob("*"))))
n_val = len(list((root / "images/val").glob("*")))
print(f"dataset root: {root}  ({n_val} val images)")

weights = next(pathlib.Path("/kaggle/input").glob("**/Text_Translation_Pipeline/best.pt"), None)
if weights is None:
    raise RuntimeError("best.pt not found under /kaggle/input")
size_mb = weights.stat().st_size / 1e6
print(f"weights: {weights}  ({size_mb:.1f} MB)")
if size_mb < 1:
    raise RuntimeError("best.pt is a git-lfs pointer, not the model")

pathlib.Path("/kaggle/working/dataset.yaml").write_text(
    f"""train: {root}/images/train
val: {root}/images/val

nc: 8

names:
  0: Arabic
  1: Latin
  2: Chinese
  3: Korean
  4: Japanese
  5: Bangla
  6: Hindi
  7: Other
"""
)
pathlib.Path("/kaggle/working/weights.txt").write_text(str(weights))

In [ ]:
import subprocess

weights = open("/kaggle/working/weights.txt").read().strip()
subprocess.run(
    ["python", "ablation/revalidate.py",
     "--weights", weights,
     "--data", "/kaggle/working/dataset.yaml",
     "--imgsz", "480", "--device", "0"],
    cwd="/kaggle/working/repo", check=True,
)

If the overall mAP50 lands near **0.663**, `best.pt` is the checkpoint behind
Fig. 11(b) and the LaTeX block above replaces Table 5 verbatim.

If it differs, `best.pt` is from a different run, and the table has to be rebuilt from
whichever checkpoint produced the figure — tell me the number and I will work out which.

One naming note: this prints class 7 as `Other`, which is what the model, `dataset.yaml`
and Fig. 11(b) all call it. The current Table 5 calls it `Symbols`.
